# T2 (assembly press-fit) data collection

Healthy and anomaly recording via RTDE at 125 Hz. Payload 1 kg.

Includes the A3 14-wrap rerun with the correct pendant.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
# Payload: ~1 kg physical, 1.0 kg pendant

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print(f"WARNING: T2 presses DOWN 7cm from HOME. Surface must be below TCP.")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_SINGLE = (90/25.4, 70/25.4)
FIG_DOUBLE = (180/25.4, 70/25.4)
FIG_TALL   = (180/25.4, 140/25.4)

C_SIGNAL = "#D55E00"
C_TCP    = "#0072B2"
C_HOME   = "#009E73"
C_FAR    = "#CC79A7"

ROOT        = Path(r"D:\Research\RCM")
LAB_DATA    = ROOT / "Lab_Data" / "T2_Assembly" / "healthy"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LAB_DATA, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB2_T2_Healthy"
TASK     = "T2"
HZ       = 125
N_SCRIPT = 180
N_ACCEPT = 160
N_WARMUP = 20

PAYLOAD_PENDANT_KG  = 1.0
PAYLOAD_PHYSICAL_KG = 1.0

# T2: slow assembly with press phases
V_FAST  = 0.05     # travel speed (m/s)
V_SLOW  = 0.02     # approach speed
V_PRESS = 0.008    # press speed (very slow — high torque)
A       = 0.15     # low accel (smooth contact)
SLEEP   = 0.4      # dwell at press

SCRIPT_PORT = 30002

def send_urscript(script: str):
    if not script.endswith("\n"):
        script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

def build_t2(N):
    return f"""
def task_t2():
  local vf = {V_FAST}
  local vs = {V_SLOW}
  local vp = {V_PRESS}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above    = pose_trans(home, p[0.08, 0.0, 0.0, 0,0,0])
  local pick          = pose_trans(home, p[0.08, 0.0, -0.03, 0,0,0])

  local press1_above   = pose_trans(home, p[0.05, -0.06, 0.0, 0,0,0])
  local press1_contact = pose_trans(home, p[0.05, -0.06, -0.04, 0,0,0])
  local press1_deep    = pose_trans(home, p[0.05, -0.06, -0.07, 0,0,0])

  local press2_above   = pose_trans(home, p[0.05, 0.06, 0.0, 0,0,0])
  local press2_contact = pose_trans(home, p[0.05, 0.06, -0.04, 0,0,0])
  local press2_deep    = pose_trans(home, p[0.05, 0.06, -0.07, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)

    movel(press1_above, a=a, v=vf)
    movel(press1_contact, a=a, v=vs)
    movel(press1_deep, a=a, v=vp)
    sleep(dt)
    movel(press1_contact, a=a, v=vs)
    movel(press1_above, a=a, v=vf)

    movel(press2_above, a=a, v=vf)
    movel(press2_contact, a=a, v=vs)
    movel(press2_deep, a=a, v=vp)
    sleep(dt)
    movel(press2_contact, a=a, v=vs)
    movel(press2_above, a=a, v=vf)

    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)
    movel(home, a=a, v=vf)

    i = i + 1
  end
end
task_t2()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")

print(f"Task: {TASK} | N_SCRIPT: {N_SCRIPT} | Accept >= {N_ACCEPT}")
print(f"Payload: {PAYLOAD_PHYSICAL_KG}kg physical, {PAYLOAD_PENDANT_KG}kg pendant")
print(f"Speeds: v_fast={V_FAST}, v_slow={V_SLOW}, v_press={V_PRESS}")
print(f"Save to: {LAB_DATA}")

if N_WARMUP > 0:
    input(f"Warmup {N_WARMUP} cycles. Press Enter to start (hand on E-STOP)...")
    send_urscript(build_t2(N_WARMUP))
    print("URScript sent - robot warming up")

    t0 = time.perf_counter()
    min_warmup = N_WARMUP * 20  # T2 cycles ~35s each
    idle = 0
    while (time.perf_counter() - t0) < N_WARMUP * 45 + 60:
        if np.max(np.abs(rtde_r.getActualQd())) < 0.001:
            idle += 1
        else:
            idle = 0
        if idle > 125 * 4 and (time.perf_counter() - t0) > min_warmup:
            break
        time.sleep(1/125)
    print(f"Warmup done ({time.perf_counter()-t0:.0f}s)")
    time.sleep(2)
else:
    print("Warmup skipped")

input(f"Collect {N_SCRIPT} cycles. Press Enter to start (E-STOP ready!)...")

dt = 1.0 / HZ
max_sec = N_SCRIPT * 45 + 120  # T2 is slower: ~35s/cycle, generous buffer
nmax = int(max_sec * HZ)

t    = np.zeros(nmax)
q    = np.zeros((nmax, 6))
qd   = np.zeros((nmax, 6))
cur  = np.zeros((nmax, 6))
tcp  = np.zeros((nmax, 6))
qdd  = np.zeros((nmax, 6))
mom  = np.zeros((nmax, 6))
tcpf = np.zeros((nmax, 6))

print("Logging started; sending URScript in 1.5s...")
t0 = time.perf_counter()
time.sleep(1.5)
send_urscript(build_t2(N_SCRIPT))
print("URScript sent. Robot should move now.\n")

idle_ctr = 0
idle_need = int(HZ * 4)
min_run = 30.0
last_print = t0

i = 0
while i < nmax:
    now = time.perf_counter()
    t[i]   = now - t0
    q[i]   = rtde_r.getActualQ()
    qd[i]  = rtde_r.getActualQd()
    cur[i] = rtde_r.getActualCurrent()
    tcp[i] = rtde_r.getActualTCPPose()
    try:    qdd[i]  = rtde_r.getTargetQdd()
    except: pass
    try:    mom[i]  = rtde_r.getTargetMoment()
    except: pass
    try:    tcpf[i] = rtde_r.getActualTCPForce()
    except: pass

    if now - last_print > 15.0:
        elapsed = now - t0
        v_now = np.max(np.abs(qd[max(0, i-HZ):i+1]))
        i_now = np.max(np.abs(cur[max(0, i-HZ):i+1]))
        print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A")
        last_print = now

    if t[i] > min_run:
        if np.max(np.abs(qd[i])) < 0.001:
            idle_ctr += 1
        else:
            idle_ctr = 0
        if idle_ctr >= idle_need:
            i += 1
            break

    target = (i + 1) * dt
    while (time.perf_counter() - t0) < target:
        pass
    i += 1

time.sleep(1)
send_urscript("stopj(2.0)\n")
print("Stop command sent.")

t, q, qd, cur, tcp = t[:i], q[:i], qd[:i], cur[:i], tcp[:i]
qdd, mom, tcpf = qdd[:i], mom[:i], tcpf[:i]

print(f"\nCollected {i:,} samples in {t[-1]:.1f}s ({i/t[-1]:.0f} Hz)")
print(f"v_max = {np.max(np.abs(qd)):.4f} rad/s | I_max = {np.max(np.abs(cur)):.3f} A")

home = tcp[0, :3]
dist = np.sqrt(np.sum((tcp[:, :3] - home)**2, axis=1)) * 1000

n_base = min(int(2.0 * HZ), len(dist) // 10)
noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
home_r = max(10.0, 3.0 * noise)
far_r  = 30.0

cycle_num = np.zeros(len(tcp), dtype=np.int32)
c = 0
was_far = False
for j in range(len(tcp)):
    if dist[j] > far_r:
        was_far = True
    if dist[j] < home_r and was_far:
        c += 1
        was_far = False
    cycle_num[j] = c

print(f"Detected {c} cycles (accept >= {N_ACCEPT}), home radius: {home_r:.1f}mm")

has_qdd  = bool(np.any(qdd != 0))
has_mom  = bool(np.any(mom != 0))
has_tcpf = bool(np.any(tcpf != 0))
print(f"target_qdd: {'available' if has_qdd else 'zeros (CB3 limitation)'}")
print(f"target_moment: {'available' if has_mom else 'zeros (CB3 limitation)'}")
print(f"actual_TCP_force: {'available' if has_tcpf else 'zeros (no F/T sensor)'}")

ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"UR5_T2_healthy_{c}cyc_{HZ}hz_{ts_str}.h5"
fpath = LAB_DATA / fname

with h5py.File(str(fpath), "w") as f:
    for name, arr in [("timestamp", t), ("actual_q", q), ("actual_qd", qd),
                      ("actual_current", cur), ("actual_TCP_pose", tcp),
                      ("target_qdd", qdd), ("target_moment", mom),
                      ("actual_TCP_force", tcpf),
                      ("cycle_number", cycle_num)]:
        f.create_dataset(name, data=arr, compression="gzip")
    f.attrs.update({
        "task": TASK, "condition": "healthy",
        "n_script": N_SCRIPT, "n_detected": c,
        "hz": HZ, "duration_sec": float(t[-1]),
        "payload_pendant_kg": PAYLOAD_PENDANT_KG,
        "payload_physical_kg": PAYLOAD_PHYSICAL_KG,
        "v_fast": V_FAST, "v_slow": V_SLOW, "v_press": V_PRESS,
        "a": A, "notebook": NOTEBOOK,
        "has_target_qdd": has_qdd,
        "has_target_moment": has_mom,
        "has_actual_TCP_force": has_tcpf,
        "home_radius_mm": float(home_r),
    })

print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

log = {
    "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
    "task": TASK, "condition": "healthy",
    "n_script": N_SCRIPT, "n_detected": c, "hz": HZ,
    "duration_sec": round(float(t[-1]), 1),
    "payload_pendant_kg": PAYLOAD_PENDANT_KG,
    "payload_physical_kg": PAYLOAD_PHYSICAL_KG,
    "tcp_home": [round(x, 4) for x in tcp[0].tolist()],
    "robot_mode": int(rtde_r.getRobotMode()),
    "safety_mode": int(rtde_r.getSafetyMode()),
    "has_target_qdd": has_qdd, "has_target_moment": has_mom,
    "has_actual_TCP_force": has_tcpf,
    "file": fname,
}
log_path = LOGS_DIR / f"{NOTEBOOK}_{ts_str}.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

csv_exists = DATASET_LOG.exists()
with open(DATASET_LOG, "a", newline="") as f:
    w = csv.writer(f)
    if not csv_exists:
        w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                     "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                     "date","notebook","pass_fail"])
    w.writerow([fname, TASK, "healthy", "none", N_SCRIPT, c,
                round(t[-1],1), HZ, PAYLOAD_PENDANT_KG, PAYLOAD_PHYSICAL_KG,
                datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                "PASS" if c >= N_ACCEPT else "FAIL"])

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]
t_rel = t - t[0]

print(f"Duration:  {t_rel[-1]:.1f}s ({t_rel[-1]/60:.1f} min)")
print(f"Samples:   {len(t):,}")
print(f"Eff. Hz:   {len(t)/t_rel[-1]:.0f}")
print(f"Cycles:    {c}")

dt_ms = np.diff(t_rel) * 1000
print(f"dt jitter: median={np.median(dt_ms):.2f}ms  p95={np.percentile(dt_ms,95):.2f}ms  "
      f"p99={np.percentile(dt_ms,99):.2f}ms  max={np.max(dt_ms):.2f}ms")

has_nan = any(np.any(np.isnan(arr)) for arr in [q, qd, cur])
print(f"NaN: {'YES' if has_nan else 'None'}")

print(f"\nJoint currents:")
for j in range(6):
    cc = cur[:, j]
    print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
          f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

print(f"\nJoint ranges:")
for j in range(6):
    rng = np.degrees(np.max(q[:,j]) - np.min(q[:,j]))
    print(f"  {JOINT[j]:16s}: {rng:.1f} deg")

# T2-specific: check press phase torque
j2_std = np.std(cur[:, 2])
if j2_std > 0.3:
    print(f"\nElbow (J2) std={j2_std:.3f}A — good press force variation")
else:
    print(f"\nElbow (J2) std={j2_std:.3f}A — press phase may be too gentle")

issues = []
if c < N_ACCEPT: issues.append(f"Cycles: {c} < {N_ACCEPT}")
if has_nan:       issues.append("NaN values")
active = sum(1 for j in range(6) if np.degrees(np.max(q[:,j])-np.min(q[:,j])) > 5)
if active < 3:    issues.append(f"Only {active} joints active")

if issues:
    print(f"\nFAIL: {'; '.join(issues)} -> rerun")
else:
    print(f"\nPASS: {c} cycles, {len(t)/t_rel[-1]:.0f} Hz, {active} active joints")

# Fig S5: T2 joint currents (first 120s — T2 cycles are longer)
mask = t_rel < 120
fig, axes = plt.subplots(6, 1, figsize=FIG_TALL, sharex=True)
fig.suptitle("T2 healthy: joint motor currents (first 120 s)", fontweight="bold")
for j in range(6):
    axes[j].plot(t_rel[mask], cur[mask,j], lw=0.3, color=C_SIGNAL, alpha=0.8)
    axes[j].set_ylabel(f"{JOINT[j]}\n(A)")
    axes[j].grid(True, alpha=0.2, lw=0.3)
    axes[j].text(-0.08, 1.0, chr(97+j), transform=axes[j].transAxes,
                 fontsize=8, fontweight="bold", va="top")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout(h_pad=0.3)
save_fig(fig, "FigS_T2_healthy_currents")
plt.close(fig)

# Fig S6: TCP distance from home
fig, ax = plt.subplots(figsize=FIG_DOUBLE)
ax.plot(t_rel, dist, lw=0.2, alpha=0.7, color=C_TCP)
ax.axhline(home_r, color=C_HOME, ls="--", lw=0.5, alpha=0.8,
           label=f"Home radius: {home_r:.1f} mm")
ax.axhline(far_r, color=C_FAR, ls="--", lw=0.5, alpha=0.7,
           label=f"Far threshold: {far_r:.0f} mm")
ax.set_ylabel("Distance from home (mm)")
ax.set_xlabel("Time (s)")
ax.set_title(f"T2 healthy: TCP distance from home ({c} cycles detected)", fontweight="bold")
ax.legend(frameon=False)
ax.grid(True, alpha=0.2, lw=0.3)
plt.tight_layout()
save_fig(fig, "FigS_T2_healthy_cycle_detect")
plt.close(fig)

# Fig S7: Cycle overlay
if c >= 3:
    fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
    fig.suptitle(f"T2 healthy: cycle overlay of motor currents ({min(30,c)} cycles)",
                 fontweight="bold")
    for j in range(6):
        ax = axes[j//2, j%2]
        for ci in range(1, min(31, c+1)):
            cmask = cycle_num == ci
            if np.sum(cmask) < 10: continue
            ax.plot(cur[cmask, j], alpha=0.15, lw=0.3)
        ax.set_title(JOINT[j])
        ax.set_ylabel("Current (A)")
        ax.set_xlabel("Sample index")
        ax.grid(True, alpha=0.2, lw=0.3)
        ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="top")
    plt.tight_layout(h_pad=0.5, w_pad=0.5)
    save_fig(fig, "FigS_T2_healthy_cycle_overlay")
    plt.close(fig)

# Fig S8: 3D TCP path
fig = plt.figure(figsize=FIG_SINGLE)
ax3 = fig.add_subplot(111, projection="3d")
ss = max(1, len(tcp)//5000)
ax3.plot(tcp[::ss,0]*100, tcp[::ss,1]*100, tcp[::ss,2]*100,
         lw=0.2, alpha=0.5, color=C_TCP)
ax3.scatter(*tcp[0,:3]*100, c=C_HOME, s=30, label="Home", zorder=5, edgecolors="none")
ax3.set_xlabel("x (cm)")
ax3.set_ylabel("y (cm)")
ax3.set_zlabel("z (cm)")
ax3.set_title("T2 healthy: TCP trajectory", fontweight="bold")
ax3.legend(frameon=False)
ax3.tick_params(pad=1)
plt.tight_layout()
save_fig(fig, "FigS_T2_healthy_tcp_3d")
plt.close(fig)

print(f"Figures saved: {FIG_SUP}")
print(f"Data saved: {fpath}")

In [ ]:
# Baseline: 1 kg gripper, 1.0 kg pendant (same as NB2 healthy)
# Anomalies: A2 (added mass), A3 (friction), A4 (obstacle), A5 (TCP offset)
# A1 skipped (proven undetectable on UR5 CB3)
# Total: 11 conditions × 40 cycles = 440 cycles, ~4.5 hrs + changeover pauses
# A3 tape reference from NB4: ~10 wraps=ok, ~17=ok, ~29=protective stop
# T2 presses harder → conservative tape levels: ~5 wraps, ~10 wraps

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print(f"T2 presses DOWN 7cm from HOME. Surface must be below TCP.")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_TALL = (180/25.4, 140/25.4)

ROOT        = Path(r"D:\Research\RCM")
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB5_T2_Anomaly"
TASK     = "T2"
HZ       = 125
SCRIPT_PORT = 30002

V_FAST  = 0.05
V_SLOW  = 0.02
V_PRESS = 0.008
A       = 0.15
SLEEP   = 0.4

def send_urscript(script: str):
    if not script.endswith("\n"):
        script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

def build_t2(N):
    return f"""
def task_t2():
  local vf = {V_FAST}
  local vs = {V_SLOW}
  local vp = {V_PRESS}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above    = pose_trans(home, p[0.08, 0.0, 0.0, 0,0,0])
  local pick          = pose_trans(home, p[0.08, 0.0, -0.03, 0,0,0])

  local press1_above   = pose_trans(home, p[0.05, -0.06, 0.0, 0,0,0])
  local press1_contact = pose_trans(home, p[0.05, -0.06, -0.04, 0,0,0])
  local press1_deep    = pose_trans(home, p[0.05, -0.06, -0.07, 0,0,0])

  local press2_above   = pose_trans(home, p[0.05, 0.06, 0.0, 0,0,0])
  local press2_contact = pose_trans(home, p[0.05, 0.06, -0.04, 0,0,0])
  local press2_deep    = pose_trans(home, p[0.05, 0.06, -0.07, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)

    movel(press1_above, a=a, v=vf)
    movel(press1_contact, a=a, v=vs)
    movel(press1_deep, a=a, v=vp)
    sleep(dt)
    movel(press1_contact, a=a, v=vs)
    movel(press1_above, a=a, v=vf)

    movel(press2_above, a=a, v=vf)
    movel(press2_contact, a=a, v=vs)
    movel(press2_deep, a=a, v=vp)
    sleep(dt)
    movel(press2_contact, a=a, v=vs)
    movel(press2_above, a=a, v=vf)

    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)
    movel(home, a=a, v=vf)

    i = i + 1
  end
end
task_t2()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]

# T2 baseline: 1kg gripper + 1kg pendant
# A2: extra mass ON TOP of 1kg baseline, pendant stays 1kg
# A3: only 2 levels (T2 presses harder, protective stop risk)
# A4: should work better on T2 (sustained press contact)
CONDITIONS = [
    # A2: Added mass — extra weight gripped alongside baseline 1kg
    {"anomaly": "A2", "severity": "1.5kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.5,
     "instruction": "A2 ADDED MASS — GRIPPER (+0.5 kg extra):\n"
                    "  - Grip baseline 1kg PLUS 0.5kg extra (total ~1.5 kg in gripper)\n"
                    "  - Pendant stays 1.0 kg (mismatch = 0.5 kg undeclared)"},

    {"anomaly": "A2", "severity": "2kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 2.0,
     "instruction": "A2 ADDED MASS — GRIPPER (+1 kg extra):\n"
                    "  - Swap: grip baseline 1kg PLUS 1kg extra (total ~2 kg)\n"
                    "  - Pendant stays 1.0 kg (mismatch = 1 kg undeclared)"},

    {"anomaly": "A2", "severity": "3kg_gripper", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 3.0,
     "instruction": "A2 ADDED MASS — GRIPPER (+2 kg extra):\n"
                    "  - Swap: grip baseline 1kg PLUS 2kg extra (total ~3 kg)\n"
                    "  - Pendant stays 1.0 kg (mismatch = 2 kg undeclared)"},

    # A3: Friction — tape on J2 (2 levels only, T2 presses hard)
    # NB4 reference: 10 wraps ok, 17 ok, 29 protective stop
    # T2 is slower but presses down — be conservative
    {"anomaly": "A3", "severity": "light_tape", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A3 FRICTION (light tape):\n"
                    "  - Remove extra weights, back to 1 kg only in gripper\n"
                    "  - Wrap ~5 rounds of tape around J2 (elbow)\n"
                    "  - NB4 reference: 10 wraps was fine on T1"},

    {"anomaly": "A3", "severity": "medium_tape", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A3 FRICTION (medium tape):\n"
                    "  - Add more tape: total ~10 rounds on J2\n"
                    "  - DO NOT go higher — T2 presses hard, protective stop risk\n"
                    "  - If robot struggles or stops, remove some tape immediately"},

    # A4: Obstacle in press path — T2 should show this better than T1
    {"anomaly": "A4", "severity": "soft", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A4 OBSTACLE (soft):\n"
                    "  - Remove all tape from J2\n"
                    "  - Place soft obstacle (sponge/foam) UNDER one press location\n"
                    "  - Robot will press DOWN onto it during press phase\n"
                    "  - This should show sustained contact force (unlike T1)"},

    {"anomaly": "A4", "severity": "medium", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A4 OBSTACLE (medium):\n"
                    "  - Swap to medium obstacle (folded cardboard/plastic block)\n"
                    "  - Same location under press path\n"
                    "  - Robot presses down onto it"},

    {"anomaly": "A4", "severity": "hard", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A4 OBSTACLE (hard):\n"
                    "  - Swap to hard obstacle (wood block/metal plate)\n"
                    "  - CAUTION: T2 presses at v=0.008 — may trigger protective stop\n"
                    "  - Hand on E-STOP. Try 5 cycles first if unsure"},

    # A5: TCP offset — grip extension piece alongside 1kg weight
    {"anomaly": "A5", "severity": "20mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A5 TCP OFFSET (20 mm):\n"
                    "  - Remove obstacle\n"
                    "  - Grip short object (~20mm sticking out below fingertips)\n"
                    "  - Keep 1 kg weight in gripper too if possible\n"
                    "  - Do NOT update TCP in pendant"},

    {"anomaly": "A5", "severity": "50mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A5 TCP OFFSET (50 mm):\n"
                    "  - Swap to ~50mm object in gripper\n"
                    "  - Do NOT update TCP in pendant"},

    {"anomaly": "A5", "severity": "100mm", "n_script": 40, "n_accept": 30,
     "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
     "instruction": "A5 TCP OFFSET (100 mm):\n"
                    "  - Swap to ~100mm object in gripper\n"
                    "  - Do NOT update TCP in pendant\n"
                    "  - CAUTION: presses 7cm + 10cm offset = may hit surface hard"},
]

print(f"NB5: {TASK} anomaly collection — {len(CONDITIONS)} conditions, "
      f"{sum(c['n_script'] for c in CONDITIONS)} total cycles")
print(f"Baseline: 1kg gripper + 1kg pendant (same as NB2 healthy)")

results = []

for idx, cond in enumerate(CONDITIONS):
    anom     = cond["anomaly"]
    sev      = cond["severity"]
    n_script = cond["n_script"]
    n_accept = cond["n_accept"]
    tag      = f"{anom}_{sev}"

    out_dir = ROOT / "Lab_Data" / "T2_Assembly" / anom
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  [{idx+1}/{len(CONDITIONS)}] {TASK} — {anom} ({sev})")
    print(f"  Cycles: {n_script} (accept >= {n_accept})")
    print(f"{'='*60}")
    print(f"\n{cond['instruction']}\n")

    input(f"Setup done? Press Enter to collect {tag} (E-STOP ready!)...")

    dt = 1.0 / HZ
    max_sec = n_script * 45 + 120
    nmax = int(max_sec * HZ)

    t_arr    = np.zeros(nmax)
    q_arr    = np.zeros((nmax, 6))
    qd_arr   = np.zeros((nmax, 6))
    cur_arr  = np.zeros((nmax, 6))
    tcp_arr  = np.zeros((nmax, 6))
    qdd_arr  = np.zeros((nmax, 6))
    mom_arr  = np.zeros((nmax, 6))
    tcpf_arr = np.zeros((nmax, 6))

    print("Logging started; sending URScript in 1.5s...")
    t0 = time.perf_counter()
    time.sleep(1.5)
    send_urscript(build_t2(n_script))
    print("URScript sent. Robot moving.\n")

    idle_ctr = 0
    idle_need = int(HZ * 4)
    min_run = 30.0
    last_print = t0

    i = 0
    while i < nmax:
        now = time.perf_counter()
        t_arr[i]   = now - t0
        q_arr[i]   = rtde_r.getActualQ()
        qd_arr[i]  = rtde_r.getActualQd()
        cur_arr[i] = rtde_r.getActualCurrent()
        tcp_arr[i] = rtde_r.getActualTCPPose()
        try:    qdd_arr[i]  = rtde_r.getTargetQdd()
        except: pass
        try:    mom_arr[i]  = rtde_r.getTargetMoment()
        except: pass
        try:    tcpf_arr[i] = rtde_r.getActualTCPForce()
        except: pass

        if now - last_print > 15.0:
            elapsed = now - t0
            v_now = np.max(np.abs(qd_arr[max(0, i-HZ):i+1]))
            i_now = np.max(np.abs(cur_arr[max(0, i-HZ):i+1]))
            print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A")
            last_print = now

        if t_arr[i] > min_run:
            if np.max(np.abs(qd_arr[i])) < 0.001:
                idle_ctr += 1
            else:
                idle_ctr = 0
            if idle_ctr >= idle_need:
                i += 1
                break

        target = (i + 1) * dt
        while (time.perf_counter() - t0) < target:
            pass
        i += 1

    time.sleep(1)
    send_urscript("stopj(2.0)\n")

    t_arr, q_arr, qd_arr = t_arr[:i], q_arr[:i], qd_arr[:i]
    cur_arr, tcp_arr = cur_arr[:i], tcp_arr[:i]
    qdd_arr, mom_arr, tcpf_arr = qdd_arr[:i], mom_arr[:i], tcpf_arr[:i]

    print(f"Collected {i:,} samples in {t_arr[-1]:.1f}s ({i/t_arr[-1]:.0f} Hz)")

    # Cycle detection
    home = tcp_arr[0, :3]
    dist = np.sqrt(np.sum((tcp_arr[:, :3] - home)**2, axis=1)) * 1000

    n_base = min(int(2.0 * HZ), len(dist) // 10)
    noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
    home_r = max(10.0, 3.0 * noise)
    far_r  = 30.0

    cycle_num = np.zeros(len(tcp_arr), dtype=np.int32)
    cyc = 0
    was_far = False
    for j in range(len(tcp_arr)):
        if dist[j] > far_r:
            was_far = True
        if dist[j] < home_r and was_far:
            cyc += 1
            was_far = False
        cycle_num[j] = cyc

    print(f"Detected {cyc} cycles (accept >= {n_accept})")

    has_qdd  = bool(np.any(qdd_arr != 0))
    has_mom  = bool(np.any(mom_arr != 0))
    has_tcpf = bool(np.any(tcpf_arr != 0))

    # Save HDF5
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"UR5_T2_{anom}_{sev}_{cyc}cyc_{HZ}hz_{ts_str}.h5"
    fpath = out_dir / fname

    with h5py.File(str(fpath), "w") as f:
        for name, arr in [("timestamp", t_arr), ("actual_q", q_arr), ("actual_qd", qd_arr),
                          ("actual_current", cur_arr), ("actual_TCP_pose", tcp_arr),
                          ("target_qdd", qdd_arr), ("target_moment", mom_arr),
                          ("actual_TCP_force", tcpf_arr),
                          ("cycle_number", cycle_num)]:
            f.create_dataset(name, data=arr, compression="gzip")
        f.attrs.update({
            "task": TASK, "condition": "anomaly",
            "anomaly_type": anom, "severity": sev,
            "n_script": n_script, "n_detected": cyc,
            "hz": HZ, "duration_sec": float(t_arr[-1]),
            "payload_pendant_kg": cond["payload_pendant_kg"],
            "payload_physical_kg": cond["payload_physical_kg"],
            "v_fast": V_FAST, "v_slow": V_SLOW, "v_press": V_PRESS,
            "a": A, "notebook": NOTEBOOK,
            "has_target_qdd": has_qdd,
            "has_target_moment": has_mom,
            "has_actual_TCP_force": has_tcpf,
            "home_radius_mm": float(home_r),
        })

    print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

    # Session log
    log = {
        "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
        "task": TASK, "condition": "anomaly",
        "anomaly_type": anom, "severity": sev,
        "n_script": n_script, "n_detected": cyc, "hz": HZ,
        "duration_sec": round(float(t_arr[-1]), 1),
        "payload_pendant_kg": cond["payload_pendant_kg"],
        "payload_physical_kg": cond["payload_physical_kg"],
        "tcp_home": [round(x, 4) for x in tcp_arr[0].tolist()],
        "robot_mode": int(rtde_r.getRobotMode()),
        "safety_mode": int(rtde_r.getSafetyMode()),
        "has_target_qdd": has_qdd, "has_target_moment": has_mom,
        "has_actual_TCP_force": has_tcpf,
        "file": fname,
    }
    log_path = LOGS_DIR / f"{NOTEBOOK}_{anom}_{sev}_{ts_str}.json"
    with open(log_path, "w") as fl:
        json.dump(log, fl, indent=2)

    # Dataset CSV
    csv_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as fl:
        w = csv.writer(fl)
        if not csv_exists:
            w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                         "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                         "date","notebook","pass_fail"])
        w.writerow([fname, TASK, "anomaly", f"{anom}_{sev}", n_script, cyc,
                    round(t_arr[-1],1), HZ, cond["payload_pendant_kg"],
                    cond["payload_physical_kg"],
                    datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                    "PASS" if cyc >= n_accept else "FAIL"])

    # QC
    has_nan = any(np.any(np.isnan(arr)) for arr in [q_arr, qd_arr, cur_arr])
    active = sum(1 for j in range(6) if np.degrees(np.max(q_arr[:,j])-np.min(q_arr[:,j])) > 5)
    passed = cyc >= n_accept and not has_nan and active >= 3

    print(f"\nJoint currents:")
    for j in range(6):
        cc = cur_arr[:, j]
        print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
              f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

    if passed:
        print(f"\nPASS: {cyc} cycles, {active} active joints")
    else:
        issues = []
        if cyc < n_accept: issues.append(f"Cycles: {cyc} < {n_accept}")
        if has_nan: issues.append("NaN")
        if active < 3: issues.append(f"Only {active} active joints")
        print(f"\nFAIL: {'; '.join(issues)} -> consider rerunning this condition")

    results.append({"tag": tag, "cycles": cyc, "pass": passed,
                    "I_max": float(np.max(np.abs(cur_arr))),
                    "J1_mean": float(np.mean(cur_arr[:,1])),
                    "J2_mean": float(np.mean(cur_arr[:,2]))})

    # Cycle overlay figure
    if cyc >= 3:
        fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
        fig.suptitle(f"T2 {anom} ({sev}): cycle overlay ({min(30,cyc)} cycles)",
                     fontweight="bold")
        for j in range(6):
            ax = axes[j//2, j%2]
            for ci in range(1, min(31, cyc+1)):
                cmask = cycle_num == ci
                if np.sum(cmask) < 10: continue
                ax.plot(cur_arr[cmask, j], alpha=0.15, lw=0.3)
            ax.set_title(JOINT[j])
            ax.set_ylabel("Current (A)")
            ax.set_xlabel("Sample index")
            ax.grid(True, alpha=0.2, lw=0.3)
            ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                    fontsize=8, fontweight="bold", va="top")
        plt.tight_layout(h_pad=0.5, w_pad=0.5)
        save_fig(fig, f"FigS_T2_{anom}_{sev}_cycle_overlay")
        plt.close(fig)

print(f"\n{'='*60}")
print(f"  NB5 COMPLETE — {TASK} anomaly collection")
print(f"{'='*60}")

print(f"\n{'Tag':<25s} {'Cycles':>6s} {'J1_mean':>8s} {'J2_mean':>8s} {'I_max':>6s} {'Pass':>5s}")
print("-" * 60)
for r in results:
    print(f"{r['tag']:<25s} {r['cycles']:>6d} {r['J1_mean']:>+8.3f} {r['J2_mean']:>+8.3f} "
          f"{r['I_max']:>6.2f} {'PASS' if r['pass'] else 'FAIL':>5s}")

n_pass = sum(1 for r in results if r["pass"])
print(f"\n{n_pass}/{len(results)} conditions passed")

print(f"\nNB2 healthy baseline for comparison:")
print(f"  J1=+3.537  J2=+2.655  I_max=5.10")

if n_pass == len(results):
    print("\nAll conditions passed. Ready for NB6 (T3 anomalies).")
else:
    failed = [r["tag"] for r in results if not r["pass"]]
    print(f"\nFailed: {', '.join(failed)} -> rerun those conditions manually")

In [ ]:
# NB5b_fix — Rerun T2 A3 medium_duct (14 wraps) with CORRECT pendant
# Previous run had pendant=3kg instead of 1kg — WRONG
# This run: pendant=1kg, gripper=1kg, 14 wraps duct tape on J2
# ~24 min

from rtde_receive import RTDEReceiveInterface
import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print()
print("Single condition: T2 A3 medium_duct (14 wraps)")
print("  Pendant: 1.0 kg  ← CHECK THIS!")
print("  Gripper: 1.0 kg")
print("  14 wraps DUCT TAPE on J2")

matplotlib.rcParams.update({
    "font.family": "Arial", "font.size": 6, "axes.titlesize": 7,
    "axes.labelsize": 6, "xtick.labelsize": 5, "ytick.labelsize": 5,
    "legend.fontsize": 5, "figure.titlesize": 7, "axes.linewidth": 0.5,
    "xtick.major.width": 0.4, "ytick.major.width": 0.4,
    "xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.4,
    "savefig.dpi": 1200, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

FIG_TALL = (180/25.4, 140/25.4)

ROOT        = Path(r"D:\Research\RCM")
OUT_DIR     = ROOT / "Lab_Data" / "T2_Assembly" / "A3"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [OUT_DIR, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK    = "NB5b_fix"
TASK        = "T2"
HZ          = 125
SCRIPT_PORT = 30002

V_FAST = 0.05; V_SLOW = 0.02; V_PRESS = 0.008; A = 0.15; SLEEP = 0.4

def send_urscript(script):
    if not script.endswith("\n"): script += "\n"
    with socket.create_connection((ROBOT_IP, SCRIPT_PORT), timeout=5) as s:
        s.sendall(script.encode("utf-8"))

def build_t2(N):
    return f"""
def task_t2():
  local vf = {V_FAST}
  local vs = {V_SLOW}
  local vp = {V_PRESS}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above    = pose_trans(home, p[0.08, 0.0, 0.0, 0,0,0])
  local pick          = pose_trans(home, p[0.08, 0.0, -0.03, 0,0,0])

  local press1_above   = pose_trans(home, p[0.05, -0.06, 0.0, 0,0,0])
  local press1_contact = pose_trans(home, p[0.05, -0.06, -0.04, 0,0,0])
  local press1_deep    = pose_trans(home, p[0.05, -0.06, -0.07, 0,0,0])

  local press2_above   = pose_trans(home, p[0.05, 0.06, 0.0, 0,0,0])
  local press2_contact = pose_trans(home, p[0.05, 0.06, -0.04, 0,0,0])
  local press2_deep    = pose_trans(home, p[0.05, 0.06, -0.07, 0,0,0])

  while (i < N):
    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)

    movel(press1_above, a=a, v=vf)
    movel(press1_contact, a=a, v=vs)
    movel(press1_deep, a=a, v=vp)
    sleep(dt)
    movel(press1_contact, a=a, v=vs)
    movel(press1_above, a=a, v=vf)

    movel(press2_above, a=a, v=vf)
    movel(press2_contact, a=a, v=vs)
    movel(press2_deep, a=a, v=vp)
    sleep(dt)
    movel(press2_contact, a=a, v=vs)
    movel(press2_above, a=a, v=vf)

    movel(pick_above, a=a, v=vf)
    movel(pick, a=a, v=vs)
    sleep(0.3)
    movel(pick_above, a=a, v=vs)
    movel(home, a=a, v=vf)

    i = i + 1
  end
end
task_t2()
"""

JOINT = ["Base (J0)","Shoulder (J1)","Elbow (J2)","Wrist 1 (J3)","Wrist 2 (J4)","Wrist 3 (J5)"]

print(f"\n{'='*60}")
print(f"  T2 A3 medium_duct RERUN — 14 wraps + 1kg")
print(f"  ⚠️  VERIFY: Pendant = 1.0 kg (NOT 3kg!)")
print(f"  ⚠️  VERIFY: Gripper = 1.0 kg")
print(f"  ⚠️  VERIFY: 14 wraps DUCT TAPE on J2")
print(f"{'='*60}")

input("\nAll correct? Press Enter to start (E-STOP ready!)...")

N_SCRIPT = 40
N_ACCEPT = 30

dt = 1.0 / HZ
max_sec = N_SCRIPT * 35 + 120
nmax = int(max_sec * HZ)

t_arr    = np.zeros(nmax)
q_arr    = np.zeros((nmax, 6))
qd_arr   = np.zeros((nmax, 6))
cur_arr  = np.zeros((nmax, 6))
tcp_arr  = np.zeros((nmax, 6))
qdd_arr  = np.zeros((nmax, 6))
mom_arr  = np.zeros((nmax, 6))
tcpf_arr = np.zeros((nmax, 6))

print("Logging started; sending URScript in 1.5s...")
t0 = time.perf_counter()
time.sleep(1.5)
send_urscript(build_t2(N_SCRIPT))
print("URScript sent. Robot moving.\n")

idle_ctr = 0
idle_need = int(HZ * 4)
min_run = 30.0
last_print = t0

i = 0
try:
    while i < nmax:
        now = time.perf_counter()
        t_arr[i]   = now - t0
        q_arr[i]   = rtde_r.getActualQ()
        qd_arr[i]  = rtde_r.getActualQd()
        cur_arr[i] = rtde_r.getActualCurrent()
        tcp_arr[i] = rtde_r.getActualTCPPose()
        try:    qdd_arr[i]  = rtde_r.getTargetQdd()
        except: pass
        try:    mom_arr[i]  = rtde_r.getTargetMoment()
        except: pass
        try:    tcpf_arr[i] = rtde_r.getActualTCPForce()
        except: pass

        if now - last_print > 15.0:
            elapsed = now - t0
            v_now = np.max(np.abs(qd_arr[max(0, i-HZ):i+1]))
            i_now = np.max(np.abs(cur_arr[max(0, i-HZ):i+1]))
            print(f"  {elapsed:6.1f}s | samples: {i+1:>7,} | v_max: {v_now:.4f} | I_max: {i_now:.2f}A")
            last_print = now

        if t_arr[i] > min_run:
            if np.max(np.abs(qd_arr[i])) < 0.001:
                idle_ctr += 1
            else:
                idle_ctr = 0
            if idle_ctr >= idle_need:
                i += 1
                break

        target = (i + 1) * dt
        while (time.perf_counter() - t0) < target:
            pass
        i += 1

except Exception as e:
    print(f"\n  ⚠️ INTERRUPTED: {e}")

time.sleep(1)
try: send_urscript("stopj(2.0)\n")
except: pass

t_arr, q_arr, qd_arr = t_arr[:i], q_arr[:i], qd_arr[:i]
cur_arr, tcp_arr = cur_arr[:i], tcp_arr[:i]
qdd_arr, mom_arr, tcpf_arr = qdd_arr[:i], mom_arr[:i], tcpf_arr[:i]

dur = t_arr[-1] if len(t_arr) > 0 else 0
print(f"Collected {i:,} samples in {dur:.1f}s ({i/dur:.0f} Hz)" if dur > 0 else "No samples")

# Cycle detection
home = tcp_arr[0, :3]
dist = np.sqrt(np.sum((tcp_arr[:, :3] - home)**2, axis=1)) * 1000
n_base = min(int(2.0 * HZ), len(dist) // 10)
noise = np.median(np.abs(np.diff(dist[:n_base]))) if n_base > 10 else 0.5
home_r = max(10.0, 3.0 * noise)
far_r  = 30.0

cycle_num = np.zeros(len(tcp_arr), dtype=np.int32)
cyc = 0; was_far = False
for j in range(len(tcp_arr)):
    if dist[j] > far_r: was_far = True
    if dist[j] < home_r and was_far:
        cyc += 1; was_far = False
    cycle_num[j] = cyc

print(f"Detected {cyc} cycles (accept >= {N_ACCEPT})")

has_qdd  = bool(np.any(qdd_arr != 0))
has_mom  = bool(np.any(mom_arr != 0))
has_tcpf = bool(np.any(tcpf_arr != 0))

ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"UR5_T2_A3_medium_duct_{cyc}cyc_{HZ}hz_{ts_str}.h5"
fpath = OUT_DIR / fname

with h5py.File(str(fpath), "w") as f:
    for name, arr in [("timestamp", t_arr), ("actual_q", q_arr), ("actual_qd", qd_arr),
                      ("actual_current", cur_arr), ("actual_TCP_pose", tcp_arr),
                      ("target_qdd", qdd_arr), ("target_moment", mom_arr),
                      ("actual_TCP_force", tcpf_arr), ("cycle_number", cycle_num)]:
        f.create_dataset(name, data=arr, compression="gzip")
    f.attrs.update({
        "task": TASK, "condition": "anomaly",
        "anomaly_type": "A3", "severity": "medium_duct",
        "tape_type": "duct_tape",
        "n_script": N_SCRIPT, "n_detected": cyc,
        "hz": HZ, "duration_sec": float(dur),
        "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
        "notebook": NOTEBOOK,
        "has_target_qdd": has_qdd, "has_target_moment": has_mom,
        "has_actual_TCP_force": has_tcpf,
        "home_radius_mm": float(home_r),
        "note": "rerun — previous had pendant=3kg instead of 1kg",
    })

print(f"Saved: {fpath} ({fpath.stat().st_size/1024/1024:.1f} MB)")

log = {
    "notebook": NOTEBOOK, "timestamp": datetime.now().isoformat(),
    "task": TASK, "condition": "anomaly",
    "anomaly_type": "A3", "severity": "medium_duct",
    "tape_type": "duct_tape",
    "n_script": N_SCRIPT, "n_detected": cyc, "hz": HZ,
    "duration_sec": round(float(dur), 1),
    "payload_pendant_kg": 1.0, "payload_physical_kg": 1.0,
    "tcp_home": [round(x, 4) for x in tcp_arr[0].tolist()],
    "file": fname, "note": "rerun — previous had pendant=3kg",
}
try:
    log["robot_mode"] = int(rtde_r.getRobotMode())
    log["safety_mode"] = int(rtde_r.getSafetyMode())
except: pass

log_path = LOGS_DIR / f"{NOTEBOOK}_T2_A3_medium_duct_{ts_str}.json"
with open(log_path, "w") as fl:
    json.dump(log, fl, indent=2)

csv_exists = DATASET_LOG.exists()
with open(DATASET_LOG, "a", newline="") as fl:
    w = csv.writer(fl)
    if not csv_exists:
        w.writerow(["filename","task","condition","anomaly","n_script","n_detected",
                     "duration_s","hz","payload_pendant_kg","payload_physical_kg",
                     "date","notebook","pass_fail"])
    w.writerow([fname, TASK, "anomaly", "A3_medium_duct", N_SCRIPT, cyc,
                round(dur,1), HZ, 1.0, 1.0,
                datetime.now().strftime("%Y-%m-%d %H:%M"), NOTEBOOK,
                "PASS" if cyc >= N_ACCEPT else "FAIL"])

has_nan = any(np.any(np.isnan(arr)) for arr in [q_arr, qd_arr, cur_arr])
active = sum(1 for j in range(6) if np.degrees(np.max(q_arr[:,j])-np.min(q_arr[:,j])) > 5)
passed = cyc >= N_ACCEPT and not has_nan and active >= 3

print(f"\nJoint currents:")
for j in range(6):
    cc = cur_arr[:, j]
    print(f"  {JOINT[j]:16s}: mean={np.mean(cc):+.3f}A  "
          f"std={np.std(cc):.3f}A  [{np.min(cc):+.2f}, {np.max(cc):+.2f}]A")

if passed:
    print(f"\nPASS: {cyc} cycles, {active} active joints")
    print("Experiment 1 data collection is NOW COMPLETE.")
else:
    issues = []
    if cyc < N_ACCEPT: issues.append(f"Cycles: {cyc} < {N_ACCEPT}")
    if has_nan: issues.append("NaN")
    if active < 3: issues.append(f"Only {active} active joints")
    print(f"\nFAIL: {'; '.join(issues)}")

print(f"\nCompare with previous runs:")
print(f"  T2 A3 medium_duct (wrong, pendant=3kg): J2_std=0.929")
print(f"  T2 A3 light_duct  (correct, pendant=1kg): J2_std=0.802")
print(f"  NB2 T2 healthy baseline:                   J2_std=0.601")

# Cycle overlay
if cyc >= 3:
    fig, axes = plt.subplots(3, 2, figsize=FIG_TALL)
    fig.suptitle(f"T2 A3 (medium_duct rerun): cycle overlay ({min(30,cyc)} cycles)",
                 fontweight="bold")
    for j in range(6):
        ax = axes[j//2, j%2]
        for ci in range(1, min(31, cyc+1)):
            cmask = cycle_num == ci
            if np.sum(cmask) < 10: continue
            ax.plot(cur_arr[cmask, j], alpha=0.15, lw=0.3)
        ax.set_title(JOINT[j])
        ax.set_ylabel("Current (A)")
        ax.set_xlabel("Sample index")
        ax.grid(True, alpha=0.2, lw=0.3)
        ax.text(-0.12, 1.0, chr(97+j), transform=ax.transAxes,
                fontsize=8, fontweight="bold", va="top")
    plt.tight_layout(h_pad=0.5, w_pad=0.5)
    fig.savefig(FIG_SUP / "FigS_T2_A3_medium_duct_rerun_cycle_overlay.png")
    fig.savefig(FIG_SUP / "FigS_T2_A3_medium_duct_rerun_cycle_overlay.pdf")
    plt.close(fig)